In [65]:
import pandas as pd

# Read the CSV file into a pandas DataFrame
df = pd.read_csv('/content/cleaned_noshow_data.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

,BookingID,NoShow,Branch,BookingMonth,ArrivalMonth,arrival_day,checkout_month,checkout_day,country,first_time,RoomType,price,platform,num_adults,num_children,Price_SGD,NumAdults,CheckoutDay
0,94113,0,Changi,November,June,25.0,June,27.0,Singapore,Yes,Single,SGD$ 492.98,Website,1,0.0,492.9800,1,27.0
1,86543,0,Orchard,August,November,28.0,November,29.0,Indonesia,Yes,King,SGD$ 1351.22,Website,2,0.0,1351.2200,2,29.0
2,75928,0,Changi,March,February,7.0,February,11.0,India,Yes,Single,NaN,Agent,1,0.0,923.4540,1,11.0
3,66947,1,Orchard,September,October,1.0,October,3.0,China,Yes,Single,SGD$ 666.04,Website,1,0.0,666.0400,1,3.0
4,106390,0,Orchard,March,June,20.0,June,24.0,Australia,Yes,Queen,USD$ 665.37,Website,1,0.0,898.2495,1,24.0


In [66]:
# Remove entirely empty rows
df.dropna(how='all', inplace=True)
print("Shape of DataFrame after removing entirely empty rows:", df.shape)
df.dropna(subset=['Price_SGD'], inplace=True)
# Explore columns with missing values
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]

if not missing_values.empty:
    print("\nColumns with missing values and their counts:")
    print(missing_values)
else:
    print("\nNo columns with missing values found.")

Shape of DataFrame after removing entirely empty rows: (97778, 18)

Columns with missing values and their counts:
price    24880
dtype: int64


## Feature Engineering

I will create two new features:
1.  **Lead Time (in days)**: The number of days between the booking month and the arrival month. Since only month names are available, I will assume a default day (1st) and intelligently infer the year to calculate the difference in days.
2.  **Length of Stay (in days)**: The number of days between the `Arrival` and `Checkout` dates. I will convert these columns to datetime objects and calculate their difference.

In [67]:
import calendar
import pandas as pd

# Map month names to numbers for easier date conversion
month_map = {name: num for num in range(1, 13) for name in calendar.month_name[num].split(',') if name}

# Function to create a date object from month name and day, inferring year
def create_full_date(row, month_col, day_source, base_year=2023):
    month_name = row[month_col]

    # day_source can be an integer (e.g., 1) or a column name (e.g., 'arrival_day')
    if isinstance(day_source, int):
        day = day_source
    else:
        day = row[day_source]

    month_num = month_map.get(month_name)

    if month_num is None or pd.isna(day) or day <= 0:
        return pd.NaT

    try:
        day = int(day)
        # Handle cases where day might exceed days in month
        _, days_in_month = calendar.monthrange(base_year, month_num)
        if day > days_in_month:
            day = days_in_month # Cap day at max days in month to avoid error
        return pd.Timestamp(year=base_year, month=month_num, day=day)
    except ValueError:
        return pd.NaT

# Generate BookingDate_temp using BookingMonth and the first day of the month
df['BookingDate_temp'] = df.apply(lambda row: create_full_date(row, 'BookingMonth', 1, base_year=2023), axis=1)

# Generate ArrivalDate_temp using ArrivalMonth and the 'arrival_day' column
df['ArrivalDate_temp'] = df.apply(lambda row: create_full_date(row, 'ArrivalMonth', 'arrival_day', base_year=2023), axis=1)

# Intelligent year inference for ArrivalDate_temp relative to BookingDate_temp
# If ArrivalDate_temp is earlier than BookingDate_temp, assume it's in the next year
df['ArrivalDate_temp'] = df.apply(
    lambda row:
        row['ArrivalDate_temp'].replace(year=row['ArrivalDate_temp'].year + 1)
        if pd.notna(row['BookingDate_temp']) and pd.notna(row['ArrivalDate_temp']) and row['ArrivalDate_temp'] < row['BookingDate_temp']
        else row['ArrivalDate_temp'],
    axis=1
)

# Calculate Lead Time (in days)
df['Lead_Time_Days'] = (df['ArrivalDate_temp'] - df['BookingDate_temp']).dt.days

# Display the new features and head of the DataFrame
display(df[['BookingMonth', 'BookingDate_temp', 'ArrivalMonth', 'arrival_day', 'ArrivalDate_temp', 'Lead_Time_Days']].head())

,BookingMonth,BookingDate_temp,ArrivalMonth,arrival_day,ArrivalDate_temp,Lead_Time_Days
0,November,2023-11-01,June,25.0,2024-06-25,237
1,August,2023-08-01,November,28.0,2023-11-28,119
2,March,2023-03-01,February,7.0,2024-02-07,343
3,September,2023-09-01,October,1.0,2023-10-01,30
4,March,2023-03-01,June,20.0,2023-06-20,111


calculate stay in days

In [68]:
import calendar
import pandas as pd

# Reuse month_map (or define it again for self-containment)
month_map = {name: num for num in range(1, 13) for name in calendar.month_name[num].split(',') if name}

# Function to create a date object from month name and day, inferring year
def create_full_date(row, month_col, day_col, base_year=2023):
    month_name = row[month_col]
    day = row[day_col]
    month_num = month_map.get(month_name)

    if month_num is None or pd.isna(day) or day <= 0:
        return pd.NaT

    try:
        # Ensure day is an integer
        day = int(day)
        # Handle cases where day might exceed days in month
        _, days_in_month = calendar.monthrange(base_year, month_num)
        if day > days_in_month:
            day = days_in_month # Cap day at max days in month to avoid error
        return pd.Timestamp(year=base_year, month=month_num, day=day)
    except ValueError:
        return pd.NaT

# The ArrivalDate_temp is already created in a previous cell (fc271d61)
# We will now create checkoutDate_temp
df['checkoutDate_temp'] = df.apply(lambda row: create_full_date(row, 'checkout_month', 'CheckoutDay', base_year=2023), axis=1)

# Intelligent year inference for checkoutDate_temp relative to ArrivalDate_temp
# If checkoutDate_temp is earlier than ArrivalDate_temp, assume it's in the next year
df['checkoutDate_temp'] = df.apply(
    lambda row:
        row['checkoutDate_temp'].replace(year=row['checkoutDate_temp'].year + 1)
        if pd.notna(row['ArrivalDate_temp']) and pd.notna(row['checkoutDate_temp']) and row['checkoutDate_temp'] < row['ArrivalDate_temp']
        else row['checkoutDate_temp'],
    axis=1
)

# Calculate Length of Stay (in days) using ArrivalDate_temp and checkoutDate_temp
df['Length_of_Stay_Days'] = (df['checkoutDate_temp'] - df['ArrivalDate_temp']).dt.days

# Display the new features and head of the DataFrame
display(df[['ArrivalMonth', 'ArrivalDate_temp',
            'checkout_month', 'checkout_day', 'checkoutDate_temp',
            'Length_of_Stay_Days']].head())

,ArrivalMonth,ArrivalDate_temp,checkout_month,checkout_day,checkoutDate_temp,Length_of_Stay_Days
0,June,2024-06-25,June,27.0,2024-06-27,2
1,November,2023-11-28,November,29.0,2023-11-29,1
2,February,2024-02-07,February,11.0,2024-02-11,4
3,October,2023-10-01,October,3.0,2023-10-03,2
4,June,2023-06-20,June,24.0,2023-06-24,4


In [69]:
df.columns

Index(['BookingID', 'NoShow', 'Branch', 'BookingMonth', 'ArrivalMonth',
       'arrival_day', 'checkout_month', 'checkout_day', 'country',
       'first_time', 'RoomType', 'price', 'platform', 'num_adults',
       'num_children', 'Price_SGD', 'NumAdults', 'CheckoutDay',
       'BookingDate_temp', 'ArrivalDate_temp', 'Lead_Time_Days',
       'checkoutDate_temp', 'Length_of_Stay_Days'],
      dtype='object')

In [72]:
df[df['Lead_Time_Days'].isnull()].head(3)

,BookingID,NoShow,Branch,BookingMonth,ArrivalMonth,arrival_day,checkout_month,checkout_day,country,first_time,...,num_adults,num_children,Price_SGD,NumAdults,CheckoutDay,BookingDate_temp,ArrivalDate_temp,Lead_Time_Days,checkoutDate_temp,Length_of_Stay_Days


In [73]:
fea_table = df[['NoShow', 'Branch', 'country',
       'first_time', 'RoomType', 'platform',
       'num_children', 'Price_SGD', 'NumAdults',
      'Lead_Time_Days','Length_of_Stay_Days']]

fea_table.head(3)

,NoShow,Branch,country,first_time,RoomType,platform,num_children,Price_SGD,NumAdults,Lead_Time_Days,Length_of_Stay_Days
0,0,Changi,Singapore,Yes,Single,Website,0.0,492.980,1,237,2
1,0,Orchard,Indonesia,Yes,King,Website,0.0,1351.220,2,119,1
2,0,Changi,India,Yes,Single,Agent,0.0,923.454,1,343,4


在fea_table的数据上跑EDA notebook:no_show率按branch、country、platform、first_time交叉对比(这就是题目要的cohort分析);用Price_SGD和length of stay做一个简化版CLV估算;用no_show概率分branch/country/room做风险评分;

## EDA on `fea_table`

### 1. No-Show Rate by Cohort (Branch, Country, Platform, First-Time)

In [74]:
# Calculate no-show rate by Branch
noshow_rate_branch = fea_table.groupby('Branch')['NoShow'].mean().reset_index()
noshow_rate_branch.rename(columns={'NoShow': 'NoShow_Rate'}, inplace=True)
display(noshow_rate_branch.sort_values(by='NoShow_Rate', ascending=False))

# Calculate no-show rate by Country
noshow_rate_country = fea_table.groupby('country')['NoShow'].mean().reset_index()
noshow_rate_country.rename(columns={'NoShow': 'NoShow_Rate'}, inplace=True)
display(noshow_rate_country.sort_values(by='NoShow_Rate', ascending=False))

# Calculate no-show rate by Platform
noshow_rate_platform = fea_table.groupby('platform')['NoShow'].mean().reset_index()
noshow_rate_platform.rename(columns={'NoShow': 'NoShow_Rate'}, inplace=True)
display(noshow_rate_platform.sort_values(by='NoShow_Rate', ascending=False))

# Calculate no-show rate by First-Time customer status
noshow_rate_first_time = fea_table.groupby('first_time')['NoShow'].mean().reset_index()
noshow_rate_first_time.rename(columns={'NoShow': 'NoShow_Rate'}, inplace=True)
display(noshow_rate_first_time.sort_values(by='NoShow_Rate', ascending=False))

,Branch,NoShow_Rate
0,Changi,0.418211
1,Orchard,0.277498


,country,NoShow_Rate
1,China,0.567534
5,Malaysia,0.353227
3,Indonesia,0.265223
6,Singapore,0.238263
2,India,0.221599
0,Australia,0.200902
4,Japan,0.171634


,platform,NoShow_Rate
1,Email,0.376132
3,Website,0.370490
0,Agent,0.368336
2,Phone,0.363657


,first_time,NoShow_Rate
1,Yes,0.378518
0,No,0.144136


### 2. Simplified CLV Estimation

In [75]:
# For a simplified CLV, we can consider the total revenue generated per booking.
# Assuming Price_SGD is per night, a simple total revenue per booking could be:
fea_table['Estimated_Booking_Value'] = fea_table['Price_SGD'] * fea_table['Length_of_Stay_Days']

# Display summary statistics of the estimated booking value
display(fea_table['Estimated_Booking_Value'].describe())

# You could also look at the average estimated value for different cohorts
clv_by_branch = fea_table.groupby('Branch')['Estimated_Booking_Value'].mean().reset_index()
clv_by_branch.rename(columns={'Estimated_Booking_Value': 'Avg_Booking_Value'}, inplace=True)
display(clv_by_branch.sort_values(by='Avg_Booking_Value', ascending=False))

/tmp/ipykernel_631/3136320612.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fea_table['Estimated_Booking_Value'] = fea_table['Price_SGD'] * fea_table['Length_of_Stay_Days']


,Estimated_Booking_Value
count,97777.000000
mean,1979.350207
std,2274.327167
min,394.213500
25%,923.454000
50%,1677.780000
75%,1965.708000
max,73053.360000


,Branch,Avg_Booking_Value
1,Orchard,2718.143711
0,Changi,1606.531431


### 3. No-Show Probability (Risk Scoring) by Branch, Country, and Room Type

In [76]:
# Calculate no-show probability by Branch and Country
risk_score_branch_country = fea_table.groupby(['Branch', 'country'])['NoShow'].mean().reset_index()
risk_score_branch_country.rename(columns={'NoShow': 'NoShow_Probability'}, inplace=True)
display(risk_score_branch_country.sort_values(by='NoShow_Probability', ascending=False))

# Calculate no-show probability by Branch, Country, and RoomType
risk_score_detailed = fea_table.groupby(['Branch', 'country', 'RoomType'])['NoShow'].mean().reset_index()
risk_score_detailed.rename(columns={'NoShow': 'NoShow_Probability'}, inplace=True)
display(risk_score_detailed.sort_values(by='NoShow_Probability', ascending=False))

,Branch,country,NoShow_Probability
1,Changi,China,0.648479
8,Orchard,China,0.423857
5,Changi,Malaysia,0.379502
3,Changi,Indonesia,0.298937
0,Changi,Australia,0.294158
6,Changi,Singapore,0.248854
2,Changi,India,0.241975
13,Orchard,Singapore,0.203552
10,Orchard,Indonesia,0.194955
4,Changi,Japan,0.183987


,Branch,country,RoomType,NoShow_Probability
3,Changi,China,King,0.677281
19,Changi,Malaysia,Single,0.491379
41,Orchard,Japan,President Suite,0.461538
31,Orchard,China,Single,0.459459
4,Changi,China,President Suite,0.454545
30,Orchard,China,Queen,0.444910
29,Orchard,China,President Suite,0.411429
28,Orchard,China,King,0.410426
45,Orchard,Malaysia,President Suite,0.400000
13,Changi,Indonesia,Single,0.395635


## Actionable Insights and Recommendations

Based on the Exploratory Data Analysis of no-show rates across different cohorts, several patterns emerge that can inform strategies to mitigate no-shows:

### 1. High-Risk Cohorts Identified:

*   **Nationality**: Customers from **China** show the highest overall no-show rate (approx. **56.8%**). This suggests a significant risk associated with this demographic.
*   **Branch-Country Combination**: Specifically, customers from **China booking at the Changi Branch** exhibit an exceptionally high no-show probability of **64.8%**. This is the highest risk segment identified.
*   **First-Time Customers**: **First-time bookers** have a substantially higher no-show rate (approx. **37.9%**) compared to returning customers (approx. **14.4%**).

### 2. Recommendations:

*   **Targeted Deposit Policy for China (Changi Branch)**:
    Given the very high no-show rate for Chinese customers at the Changi Branch, it is strongly recommended to implement a **mandatory prepayment or deposit policy** for bookings made by this specific cohort. This could significantly reduce financial losses from no-shows.

*   **Enhanced Communication for First-Time Customers**:
    For all first-time customers, consider implementing an enhanced communication strategy. This could include:
    *   More frequent booking reminders (e.g., 7 days, 3 days, and 24 hours before arrival).
    *   Personalized welcome emails detailing the benefits of staying and the importance of confirming attendance.
    *   Offering a small, low-cost incentive for successful check-ins (e.g., a welcome drink, a discount voucher for future stay).

*   **Review Cancellation Policies for High No-Show Countries**:
    For countries with overall high no-show rates like China, Malaysia (35.3%), and Indonesia (26.5%), review and potentially adjust cancellation policies to encourage timely cancellations rather than outright no-shows. This could involve tiered cancellation fees or clearer communication of existing policies.

*   **Platform-Specific Analysis**: While platform differences were less stark, consider deeper analysis for specific platforms that might contribute to higher no-shows within high-risk cohorts. For example, if a particular platform is popular with Chinese customers at the Changi branch, investigate if that platform's booking process contributes to no-shows.

These recommendations aim to reduce no-show instances by targeting the highest-risk segments with specific, actionable strategies.

Streamlit的简单交互仪表盘(可以下钻到branch或country维度)